# 검증 02: cache_store (Redis CRUD)

redis-handler 사용법 + cache_store 모든 함수 검증.

Redis 미연결 시 메모리 fallback (프로세스 종료 시 데이터 소멸).

## 0. redis-handler 기본 사용법

In [1]:
from dotenv import load_dotenv
load_dotenv("../../../.env")


True

In [2]:
from redis_handler import RedisHandler, RedisConfig, BaseJsonStore

print("[RedisConfig]")
print(f"  설정: {RedisConfig.get_summary()}")
print(f"  configured: {RedisConfig.is_configured()}")

rh = RedisHandler()
print(f"\n[RedisHandler]")
print(f"  storage_mode: {rh.storage_mode}")  # 'redis' 또는 'memory'

[RedisConfig]
  설정: {'url': 'redis://192.7.9.45:7617/0', 'socket_timeout': 5, 'socket_connect_timeout': 5, 'env_path': '환경변수'}
  configured: True

[RedisHandler]
  storage_mode: redis


In [3]:
# RedisHandler 기본 CRUD
print("=== RedisHandler.set() ===")
result = rh.set("test:verify", "hello", ttl=60)
print(f"  [입력] key='test:verify', value='hello', ttl=60")
print(f"  [출력] {result}")

print("\n=== RedisHandler.get() ===")
val = rh.get("test:verify")
print(f"  [입력] key='test:verify'")
print(f"  [출력] '{val}'")

print("\n=== RedisHandler.ttl() ===")
ttl_val = rh.ttl("test:verify")
print(f"  [입력] key='test:verify'")
print(f"  [출력] {ttl_val}초 남음")

print("\n=== RedisHandler.exists() ===")
exists = rh.exists("test:verify")
print(f"  [입력] key='test:verify'")
print(f"  [출력] {exists}")

print("\n=== RedisHandler.delete() ===")
deleted = rh.delete("test:verify")
print(f"  [입력] key='test:verify'")
print(f"  [출력] 삭제 수: {deleted}")
print(f"  삭제 후 get: {rh.get('test:verify')}")

=== RedisHandler.set() ===
  [입력] key='test:verify', value='hello', ttl=60
  [출력] True

=== RedisHandler.get() ===
  [입력] key='test:verify'
  [출력] 'hello'

=== RedisHandler.ttl() ===
  [입력] key='test:verify'
  [출력] 60초 남음

=== RedisHandler.exists() ===
  [입력] key='test:verify'
  [출력] True

=== RedisHandler.delete() ===
  [입력] key='test:verify'
  [출력] 삭제 수: 1
  삭제 후 get: None


In [5]:
# BaseJsonStore 패턴 체험
class DemoStore(BaseJsonStore):
    def __init__(self):
        super().__init__(prefix="demo", default_ttl=10)

demo = DemoStore()

print("=== _make_key() ===")
key = demo._make_key("user", "abc")
print(f"  [입력] 'user', 'abc'")
print(f"  [출력] '{key}'")

print("\n=== _set_json() ===")
data = {"name": "테스트", "score": 95, "tags": ["a", "b"]}
result = demo._set_json(key, data)
print(f"  [입력] key='{key}', value={data}")
print(f"  [출력] {result}")

print("\n=== _get_json() ===")
loaded = demo._get_json(key)
print(f"  [입력] key='{key}'")
print(f"  [출력] {loaded}")
print(f"  타입: {type(loaded)}")

print("\n=== _exists() ===")
print(f"  [출력] {demo._exists(key)}")

print("\n=== _ttl() ===")
print(f"  [출력] {demo._ttl(key)}초")

print("\n=== _delete() ===")
# demo._delete(key)
# print(f"  삭제 후: {demo._get_json(key)}")

=== _make_key() ===
  [입력] 'user', 'abc'
  [출력] 'demo:user:abc'

=== _set_json() ===
  [입력] key='demo:user:abc', value={'name': '테스트', 'score': 95, 'tags': ['a', 'b']}
  [출력] True

=== _get_json() ===
  [입력] key='demo:user:abc'
  [출력] {'name': '테스트', 'score': 95, 'tags': ['a', 'b']}
  타입: <class 'dict'>

=== _exists() ===
  [출력] True

=== _ttl() ===
  [출력] 10초

=== _delete() ===


---
## 1. FileCacheStore 싱글톤

In [7]:
from app.logic.file_handler.cache_store import _store, FILES_API_TTL, CONTEXT_CACHE_DEFAULT_TTL

print(f"타입: {type(_store).__name__}")
print(f"prefix: {_store._prefix}")
print(f"storage_mode: {_store.storage_mode}")
print(f"FILES_API_TTL: {FILES_API_TTL}초 ({FILES_API_TTL/3600:.0f}시간)")
print(f"CONTEXT_CACHE_DEFAULT_TTL: {CONTEXT_CACHE_DEFAULT_TTL}초 ({CONTEXT_CACHE_DEFAULT_TTL/3600:.0f}시간)")
print()
print(f"키 예시: {_store._make_key('files_api', 'sha256:abc')}")
print(f"키 예시: {_store._make_key('context_cache', 'hash123')}")

타입: FileCacheStore
prefix: file_cache
storage_mode: redis
FILES_API_TTL: 172800초 (48시간)
CONTEXT_CACHE_DEFAULT_TTL: 3600초 (1시간)

키 예시: file_cache:files_api:sha256:abc
키 예시: file_cache:context_cache:hash123


---
## 2. make_context_cache_key()

In [8]:
from app.logic.file_handler.cache_store import make_context_cache_key

print("=== 기본 ===")
key1 = make_context_cache_key(["sha256:abc"], model="gemini-2.5-flash")
print(f"  [입력] checksums=['sha256:abc'], model='gemini-2.5-flash'")
print(f"  [출력] {key1}")
print(f"  길이: {len(key1)}, hex: {all(c in '0123456789abcdef' for c in key1)}")

=== 기본 ===
  [입력] checksums=['sha256:abc'], model='gemini-2.5-flash'
  [출력] 27d1f5dfcfe73f4d67a0063c7d06e07f692b69ecb0637d9cb1bccb7abf9c57e6
  길이: 64, hex: True


In [10]:
# 같은 입력 → 같은 키
key2 = make_context_cache_key(["sha256:abc"], model="gemini-2.5-flash")
print(f"동일 입력 동일 키: {key1 == key2}")

동일 입력 동일 키: True


In [12]:
# 다른 model → 다른 키
key3 = make_context_cache_key(["sha256:abc"], model="gemini-2.5-pro")
print(f"다른 model: {key1 != key3}")

다른 model: True


In [11]:
# 다른 system → 다른 키
key4 = make_context_cache_key(["sha256:abc"], model="gemini-2.5-flash", system_instruction="A")
key5 = make_context_cache_key(["sha256:abc"], model="gemini-2.5-flash", system_instruction="B")
print(f"다른 system: {key4 != key5}")

다른 system: True


In [13]:
# 다른 checksum → 다른 키
key6 = make_context_cache_key(["sha256:abc"])
key7 = make_context_cache_key(["sha256:def"])
print(f"다른 checksum: {key6 != key7}")

다른 checksum: True


In [14]:
# checksum 순서 무관
key8 = make_context_cache_key(["sha256:a", "sha256:b"])
key9 = make_context_cache_key(["sha256:b", "sha256:a"])
print(f"순서 무관: {key8 == key9}")

순서 무관: True


---
### 2-내부. make_context_cache_key() 내부 동작 확인

In [15]:
# canonicalize → sha256 과정을 한 줄씩 확인
from app.logic.keying import canonicalize_llm_input, make_cache_key

file_checksums = ["sha256:abc"]
model = "gemini-2.5-flash"
system_instruction = None

# 1) 입력 dict 구성
input_dict = {
    "file_checksums": sorted(file_checksums),
    "model": model or "",
    "system_instruction": system_instruction or "",
}
print(f"[1] input_dict = {input_dict}")

# 2) canonicalize → 결정적 JSON 문자열
canonical = canonicalize_llm_input(input_dict)
print(f"[2] canonical = {canonical}")

# 3) SHA-256 해시
hash_key = make_cache_key(canonical)
print(f"[3] hash_key = {hash_key}")
print(f"    길이: {len(hash_key)}")

[1] input_dict = {'file_checksums': ['sha256:abc'], 'model': 'gemini-2.5-flash', 'system_instruction': ''}
[2] canonical = {"file_checksums":["sha256:abc"],"model":"gemini-2.5-flash","system_instruction":""}
[3] hash_key = 27d1f5dfcfe73f4d67a0063c7d06e07f692b69ecb0637d9cb1bccb7abf9c57e6
    길이: 64


---
## 3. save_files_api() / find_files_api()

In [16]:
from app.logic.file_handler.cache_store import save_files_api, find_files_api

# 단일 파일
doc_single = {
    "file_checksum": "sha256:v_single",
    "file_name": "report.pdf",
    "file_size": 30_000_000,
    "mime_type": "application/pdf",
    "uris": ["https://example.com/files/single"],
    "split": False,
}
print("=== save_files_api() ===")
print(f"  [입력] checksum='sha256:v_single', doc={doc_single}")
save_files_api("sha256:v_single", doc_single)
print(f"  [출력] 저장 완료")

print("\n=== find_files_api() ===")
result = find_files_api("sha256:v_single")
print(f"  [입력] checksum='sha256:v_single'")
print(f"  [출력] {result}")

=== save_files_api() ===
  [입력] checksum='sha256:v_single', doc={'file_checksum': 'sha256:v_single', 'file_name': 'report.pdf', 'file_size': 30000000, 'mime_type': 'application/pdf', 'uris': ['https://example.com/files/single'], 'split': False}
  [출력] 저장 완료

=== find_files_api() ===
  [입력] checksum='sha256:v_single'
  [출력] {'file_checksum': 'sha256:v_single', 'file_name': 'report.pdf', 'file_size': 30000000, 'mime_type': 'application/pdf', 'uris': ['https://example.com/files/single'], 'split': False}


---
### 3-내부. save_files_api() / find_files_api() 내부 동작 확인

In [17]:
# split PDF (여러 URI)
doc_split = {
    "file_checksum": "sha256:v_split",
    "file_name": "large.pdf",
    "file_size": 120_000_000,
    "mime_type": "application/pdf",
    "uris": ["https://ex.com/part1", "https://ex.com/part2", "https://ex.com/part3"],
    "split": True,
}
save_files_api("sha256:v_split", doc_split)
result = find_files_api("sha256:v_split")
print(f"split: {result['split']}, uris 수: {len(result['uris'])}")
for i, uri in enumerate(result['uris']):
    print(f"  {i+1}: {uri}")

split: True, uris 수: 3
  1: https://ex.com/part1
  2: https://ex.com/part2
  3: https://ex.com/part3


In [18]:
# miss
print("=== find_files_api() miss ===")
result = find_files_api("sha256:nonexistent")
print(f"  [입력] 'sha256:nonexistent'")
print(f"  [출력] {result}")

=== find_files_api() miss ===
  [입력] 'sha256:nonexistent'
  [출력] None


---
## 4. save_context_cache() / find_context_cache()

In [19]:
from app.logic.file_handler.cache_store import save_context_cache, find_context_cache

doc_ctx = {
    "file_checksums": ["sha256:v_ctx"],
    "model": "gemini-2.5-flash",
    "system_instruction": "테스트",
    "cached_content_name": "cachedContents/verify",
    "files_api_uris": ["https://ex.com/ctx"],
}
print("=== save_context_cache() ===")
print(f"  [입력] hash_key='v_hash', doc={doc_ctx}, ttl=300")
save_context_cache("v_hash", doc_ctx, ttl=300)
print(f"  [출력] 저장 완료")

print("\n=== find_context_cache() ===")
result = find_context_cache("v_hash")
print(f"  [입력] 'v_hash'")
print(f"  [출력] {result}")

=== save_context_cache() ===
  [입력] hash_key='v_hash', doc={'file_checksums': ['sha256:v_ctx'], 'model': 'gemini-2.5-flash', 'system_instruction': '테스트', 'cached_content_name': 'cachedContents/verify', 'files_api_uris': ['https://ex.com/ctx']}, ttl=300
  [출력] 저장 완료

=== find_context_cache() ===
  [입력] 'v_hash'
  [출력] {'file_checksums': ['sha256:v_ctx'], 'model': 'gemini-2.5-flash', 'system_instruction': '테스트', 'cached_content_name': 'cachedContents/verify', 'files_api_uris': ['https://ex.com/ctx']}


In [20]:
# miss
result = find_context_cache("nonexistent")
print(f"miss: {result is None}")

miss: True


---
## 5. delete_files_api() / delete_context_cache()

In [21]:
from app.logic.file_handler.cache_store import delete_files_api, delete_context_cache

print("=== delete_files_api() ===")
print(f"  [입력] 'sha256:v_single'")
delete_files_api("sha256:v_single")
print(f"  삭제 후: {find_files_api('sha256:v_single') is None}")

delete_files_api("sha256:v_split")
print(f"  split 삭제 후: {find_files_api('sha256:v_split') is None}")

print("\n=== delete_context_cache() ===")
print(f"  [입력] 'v_hash'")
delete_context_cache("v_hash")
print(f"  삭제 후: {find_context_cache('v_hash') is None}")

=== delete_files_api() ===
  [입력] 'sha256:v_single'
  삭제 후: True
  split 삭제 후: True

=== delete_context_cache() ===
  [입력] 'v_hash'
  삭제 후: True


---
## 정리 (cleanup)

테스트 중 생성한 데이터 삭제

In [ ]:
# 테스트 데이터 정리
from app.logic.file_handler.cache_store import delete_files_api, delete_context_cache

# 메모리 백엔드 정리 (Redis 미연결 시에도 동작)
rh = RedisHandler()
cleanup_keys = [k for k in rh._memory._data.keys()]
for k in cleanup_keys:
    rh.delete(k)
print(f"정리 완료: {len(cleanup_keys)}개 키 삭제")

---
## 정리

| 함수 | 입력 | 출력 |
|------|------|------|
| `RedisHandler.set/get/ttl/exists/delete` | key, value, ttl | bool/str/int |
| `BaseJsonStore._make_key/_set_json/_get_json/_delete/_exists/_ttl` | key parts, value | str/bool/dict |
| `FileCacheStore` | prefix='file_cache' | 싱글톤 |
| `make_context_cache_key()` | checksums, model, system | 64자 hex |
| `save_files_api()` | checksum, doc, ttl | None |
| `find_files_api()` | checksum | dict or None |
| `save_context_cache()` | hash_key, doc, ttl | None |
| `find_context_cache()` | hash_key | dict or None |
| `delete_files_api()` | checksum | None |
| `delete_context_cache()` | hash_key | None |